02 Limpieza de datos - Limpiar el dataset rockyu y el de kaggle para guardar la versines limpias en data/proceesed

In [1]:
import pandas as pd

In [2]:
with open("../data/raw/rockyou.txt", "rb") as archivo:
    lineas = archivo.read().split(b"\n")

if lineas[-1] == b"":
    lineas = lineas[:-1]

print(f"Líneas reales: {len(lineas):,}")

Líneas reales: 14,344,391


In [ ]:
def decodificar(linea):
    try:
        return linea.decode("utf-8")
    except UnicodeDecodeError:
        return linea.decode("latin-1")

rockyou = pd.DataFrame({"password": [decodificar(l) for l in lineas]})
rockyou["ranking"] = range(1, len(rockyou) + 1)

print(f"Filas: {len(rockyou):,}")
rockyou.head(10)

Filas: 14,344,391


,password,ranking
0,123456,1
1,12345,2
2,123456789,3
3,password,4
4,iloveyou,5
5,princess,6
6,1234567,7
7,rockyou,8
8,12345678,9
9,abc123,10


In [5]:
s = rockyou["password"]

vacias = s.str.strip() == ""
control = s.str.contains(r"[\x00-\x1F\x7F]", regex=True)
largas = s.str.len() > 64
duplicadas = s.duplicated()

print(f"Vacias o solo espacios: {vacias.sum():,}")
print(f"Con caracteres de control: {control.sum():,}")
print(f"Con más de 64 caracteres: {largas.sum():,}")
print(f"Duplicadas: {duplicadas.sum():,}")

Vacias o solo espacios: 12
Con caracteres de control: 9
Con más de 64 caracteres: 872
Duplicadas: 36


In [6]:
s[largas].head(5)

1188492    vabfdvfdlvhjibfedblsfndilvbgilebvgdlsbgvhbesgh...
1567740    llllllllllllllllllllllllllllllllllllllllllllll...
2025600    asfunction:OpenWin,http://profile.myspace.com/...
2092558    Save puts your PhotoFX in MyStuff. You can go ...
2092559    Save puts your FunNotes in MyStuff. You can go...
Name: password, dtype: str

In [8]:
antes = len(rockyou)
rockyou_limpio = rockyou[~(vacias | control | largas | duplicadas)].copy()
despues = len(rockyou_limpio)

print(f"Antes: {antes:,}")
print(f"Después: {despues:,}")
print(f"Eliminadas: {antes - despues:,} ({(antes - despues) / antes:.4%})")

Antes: 14,344,391
Después: 14,343,462
Eliminadas: 929 (0.0065%)


In [10]:
rockyou_limpio.to_parquet("../data/processed/rockyou_limpio.parquet", index=False)
print ("Guardado")

Guardado
